# CGL Optimizer Run Guide
This notebook updates row-level CGL values in the Excel file and validates Tier|Term weighted averages against Rating Agency targets.

## What this script does
- Uses Tier + Term targets to calibrate row-level CGL.
- Applies FICO-band slope by tier.
- Applies Non-LB discount by tier (where applicable).
- Uses UPB% (col M) as weight, with fallback to Total Current Principal (col L).
- Enforces CGL floor/ceiling: **0.25% to 22.00%**.
- Writes CGL to **column I** and saves the workbook.

## Before you run
- Confirm Excel is installed on this machine.
- Ensure the file path inside the code cell is correct.
- Ensure `pywin32` is available in this notebook kernel.

## How to run
1. Run the Python code cell below this markdown cell.
2. Wait for the printed summary.

## Expected output
- `Rows updated: ...`
- `Min CGL: ...%`
- `Max CGL: ...%`
- A Tier|Term table with `target`, `actual`, and `diff` values.

In [1]:
# CGL optimizer update script (Excel COM) for Windows
# This writes row-level CGL (column I) and calibrates weighted averages to targets.
# Requirements: Windows + Excel installed, and pywin32 available in this Python environment.

import gc
import math

try:
    import win32com.client as win32
except ImportError as e:
    raise ImportError(
        "pywin32 is required. In this environment, install via corporate-approved channel, "
        "then re-run this cell."
    ) from e

FILE_PATH = r"g:\Shared drives\Capital Markets 2\Capital Markets - Dropbox\20. Projects\Loan pricing automation\CGL optimizer.xlsx"
SHEET_NAME = "Pricing Tier-Term-Fico-Channel-"

# Tier|Term targets in percentage points
TARGETS = {
    "1|24": 1.00, "1|36": 2.00, "1|48": 2.50, "1|60": 5.00, "1|72": 6.50, "1|84": 9.00,
    "2|24": 2.00, "2|36": 3.00, "2|48": 6.50, "2|60": 7.75, "2|72": 8.50, "2|84": 12.00,
    "3|24": 2.75, "3|36": 4.50, "3|48": 8.50, "3|60": 11.00, "3|72": 11.50, "3|84": 16.00,
    "4|24": 4.00, "4|36": 5.50, "4|48": 13.00, "4|60": 13.50, "4|72": 15.00, "4|84": 19.50,
    "5|24": 4.50, "5|36": 7.50, "5|48": 14.00, "5|60": 17.00,
    "6|24": 6.00, "6|36": 9.00, "6|48": 17.00, "6|60": 18.00,
}

FICO_RANK = {
    ">800": 0,
    "780-800": 1,
    "760-779": 2,
    "740-759": 3,
    "720-739": 4,
    "700-719": 5,
    "<700": 6,
}

SLOPE_BY_TIER = {1: 0.25, 2: 0.25, 3: 0.50, 4: 0.50, 5: 0.75, 6: 0.75}
NON_LB_DISCOUNT_BY_TIER = {2: 0.5, 3: 1.0, 4: 1.0, 5: 2.0, 6: 2.0}

FLOOR = 0.25
CEILING = 22.00

excel = None
wb = None
ws = None

try:
    excel = win32.Dispatch("Excel.Application")
    excel.Visible = False
    excel.DisplayAlerts = False

    wb = excel.Workbooks.Open(FILE_PATH)
    ws = wb.Worksheets(SHEET_NAME)

    last_row = ws.UsedRange.Rows.Count

    records = []
    sum_w = {}
    sum_wr = {}
    sum_wd = {}

    # Pass 1: collect records and weighted aggregates
    for r in range(2, last_row + 1):
        tier_val = ws.Cells(r, 3).Value
        term_val = ws.Cells(r, 4).Value
        if tier_val is None or term_val is None:
            continue

        tier = int(tier_val)
        term = int(term_val)
        key = f"{tier}|{term}"
        if key not in TARGETS:
            continue

        fico = str(ws.Cells(r, 5).Text or "").strip()
        if fico not in FICO_RANK:
            continue
        rank = FICO_RANK[fico]

        channel = str(ws.Cells(r, 6).Text or "").strip().lower()
        discount = 0.0
        if tier != 1 and channel == "non lb" and tier in NON_LB_DISCOUNT_BY_TIER:
            discount = NON_LB_DISCOUNT_BY_TIER[tier]

        wt = ws.Cells(r, 13).Value  # UPB% (col M)
        if wt is None or float(wt) == 0.0:
            wt = ws.Cells(r, 12).Value  # Total Current Principal (col L)
        weight = float(wt or 0.0)

        slope = SLOPE_BY_TIER[tier]

        records.append(
            {
                "row": r,
                "key": key,
                "tier": tier,
                "term": term,
                "rank": rank,
                "slope": slope,
                "discount": discount,
                "weight": weight,
            }
        )

        if key not in sum_w:
            sum_w[key] = 0.0
            sum_wr[key] = 0.0
            sum_wd[key] = 0.0

        sum_w[key] += weight
        sum_wr[key] += weight * rank
        sum_wd[key] += weight * discount

    # Calibrate base by Tier|Term so weighted avg matches target before clamping
    base_by_key = {}
    for key, target in TARGETS.items():
        if key in sum_w and sum_w[key] > 0:
            tier = int(key.split("|")[0])
            slope = SLOPE_BY_TIER[tier]
            avg_rank = sum_wr[key] / sum_w[key]
            avg_discount = sum_wd[key] / sum_w[key]
            base_by_key[key] = target - slope * avg_rank + avg_discount
        else:
            base_by_key[key] = target

    # Pass 2: write row-level CGL in col I and verify weighted actuals
    rows_updated = 0
    min_cgl = float("inf")
    max_cgl = float("-inf")
    act_w = {}
    act_wc = {}

    for rec in records:
        cgl = base_by_key[rec["key"]] + rec["slope"] * rec["rank"] - rec["discount"]
        cgl = max(FLOOR, min(CEILING, cgl))

        ws.Cells(rec["row"], 9).Value = cgl / 100.0
        ws.Cells(rec["row"], 9).NumberFormat = "0.00%"
        rows_updated += 1

        min_cgl = min(min_cgl, cgl)
        max_cgl = max(max_cgl, cgl)

        key = rec["key"]
        if key not in act_w:
            act_w[key] = 0.0
            act_wc[key] = 0.0
        act_w[key] += rec["weight"]
        act_wc[key] += rec["weight"] * cgl

    wb.Save()

    print(f"Rows updated: {rows_updated}")
    print(f"Min CGL: {min_cgl:.2f}%")
    print(f"Max CGL: {max_cgl:.2f}%")
    print("\nTier|Term validation:")

    ordered_keys = sorted(TARGETS.keys(), key=lambda k: (int(k.split("|")[0]), int(k.split("|")[1])))
    for key in ordered_keys:
        target = TARGETS[key]
        if key in act_w and act_w[key] > 0:
            actual = act_wc[key] / act_w[key]
            diff = actual - target
            print(f"{key}\ttarget={target:.2f}%\tactual={actual:.4f}%\tdiff={diff:.4f}%")
        else:
            print(f"{key}\ttarget={target:.2f}%\tactual=NA\tdiff=NA")

finally:
    if wb is not None:
        wb.Close(SaveChanges=True)
    if excel is not None:
        excel.Quit()
    ws = None
    wb = None
    excel = None
    gc.collect()

Rows updated: 418
Min CGL: 0.52%
Max CGL: 21.06%

Tier|Term validation:
1|24	target=1.00%	actual=1.0000%	diff=0.0000%
1|36	target=2.00%	actual=2.0000%	diff=0.0000%
1|48	target=2.50%	actual=2.5000%	diff=0.0000%
1|60	target=5.00%	actual=5.0000%	diff=-0.0000%
1|72	target=6.50%	actual=6.5000%	diff=0.0000%
1|84	target=9.00%	actual=9.0000%	diff=0.0000%
2|24	target=2.00%	actual=2.0000%	diff=-0.0000%
2|36	target=3.00%	actual=3.0000%	diff=0.0000%
2|48	target=6.50%	actual=6.5000%	diff=-0.0000%
2|60	target=7.75%	actual=7.7500%	diff=-0.0000%
2|72	target=8.50%	actual=8.5000%	diff=0.0000%
2|84	target=12.00%	actual=12.0000%	diff=-0.0000%
3|24	target=2.75%	actual=2.7500%	diff=0.0000%
3|36	target=4.50%	actual=4.5000%	diff=0.0000%
3|48	target=8.50%	actual=8.5000%	diff=0.0000%
3|60	target=11.00%	actual=11.0000%	diff=0.0000%
3|72	target=11.50%	actual=11.5000%	diff=0.0000%
3|84	target=16.00%	actual=16.0000%	diff=0.0000%
4|24	target=4.00%	actual=4.0000%	diff=0.0000%
4|36	target=5.50%	actual=5.5000%	diff=0.0